In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

### conf

In [ ]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf, open_dict

cfg_root = "/home/n.sorokin/verl-agent/verl/trainer/config"
with initialize_config_dir(config_dir=cfg_root, version_base=None):
    cfg = compose(
        config_name="ppo_trainer",
        overrides=[
            "data.train_files=/home/n.sorokin/data/verl-agent/text/train.parquet",
            "data.val_files=/home/n.sorokin/data/verl-agent/text/test.parquet",
            "actor_rollout_ref.model.path=Qwen/Qwen2.5-1.5B-Instruct",
            "env.env_name=craftext/CraftextEnv",
            "actor_rollout_ref.rollout.name=vllm",
        ],
    )

with open_dict(cfg):
    cfg.trainer.resume_mode = "resume_path"
    cfg.trainer.resume_from_path = "/home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500"
    cfg.trainer.val_only = True             # stop after one validation pass
    cfg.trainer.val_before_train = True     # ensure validation runs
    cfg.trainer.num_samples_to_log = 1
    cfg.trainer.logger = ["console"]
    cfg.data.val_batch_size = 1
    cfg.data.train_batch_size = 1
    cfg.env.rollout.n = 1
    cfg.actor_rollout_ref.rollout.val_kwargs.n = 1
    cfg.actor_rollout_ref.rollout.val_kwargs.do_sample = False  # deterministic
    cfg.actor_rollout_ref.actor.micro_batch_size = 1
    cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu = 1
    cfg.actor_rollout_ref.rollout.log_prob_micro_batch_size_per_gpu = 1
    cfg.actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu = 1
    cfg.algorithm.adv_estimator = "grpo"  # optional: skips the critic
    cfg.actor_rollout_ref.rollout.tensor_model_parallel_size = 1
    cfg.trainer.n_gpus_per_node = 1
    cfg.trainer.nnodes = 1
    cfg.env.craftext_settings = "achievements_wood"
    cfg.actor_rollout_ref.model.lora_rank = 64
    cfg.actor_rollout_ref.model.lora_alpha = 64
    cfg.actor_rollout_ref.model.target_modules = "all-linear"
    cfg.actor_rollout_ref.model.use_remove_padding = True
    cfg.actor_rollout_ref.actor.optim.lr = 1e-6
    cfg.actor_rollout_ref.actor.ppo_mini_batch_size = 16
    cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu = 8
    cfg.actor_rollout_ref.actor.ppo_mini_batch_size = 16
    cfg.data.return_raw_chat = True
    cfg.data.max_prompt_length = 2048
    cfg.data.max_response_length = 512

In [ ]:
import ray, torch
from verl.utils import hf_tokenizer, hf_processor
from agent_system.environments import make_envs
from agent_system.multi_turn_rollout import TrajectoryCollector
from verl.trainer.ppo.ray_trainer import RayPPOTrainer, ResourcePoolManager, Role
from verl.single_controller.ray import RayWorkerGroup
from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker

ray.init(num_cpus=cfg.ray_init.num_cpus)

# HF artifacts for the base model
tokenizer = hf_tokenizer(cfg.actor_rollout_ref.model.path, trust_remote_code=cfg.data.get("trust_remote_code", False))
processor = hf_processor(cfg.actor_rollout_ref.model.path, trust_remote_code=cfg.data.get("trust_remote_code", False), use_fast=True)

# Environments (train + val); you only need val_envs for exploration
envs, val_envs = make_envs(cfg)

traj_collector = TrajectoryCollector(config=cfg, tokenizer=tokenizer, processor=processor)

In [ ]:
from verl.trainer.ppo.ray_trainer import Role, RayClassWithInitArgs

role_worker_mapping = {
    Role.ActorRollout: ray.remote(ActorRolloutRefWorker),
    Role.Critic: ray.remote(CriticWorker),
}
resource_pool_spec = {"pool": [cfg.trainer.n_gpus_per_node] * cfg.trainer.nnodes}
resource_pool_manager = ResourcePoolManager(resource_pool_spec=resource_pool_spec, mapping={
    Role.ActorRollout: "pool",
    Role.Critic: "pool",
})

trainer = RayPPOTrainer(
    config=cfg,
    tokenizer=tokenizer,
    processor=processor,
    role_worker_mapping=role_worker_mapping,
    resource_pool_manager=resource_pool_manager,
    ray_worker_group_cls=RayWorkerGroup,
    reward_fn=None,              # rule-based EpisodeRewardManager is instantiated inside _validate
    val_reward_fn=None,
    traj_collector=traj_collector,
    envs=envs,
    val_envs=val_envs,
)
trainer.init_workers()
trainer._load_checkpoint()

In [ ]:
from verl import DataProto
from agent_system.multi_turn_rollout.utils import to_list_of_dict

val_iter = iter(trainer.val_dataloader)
batch_dict = next(val_iter)                                   # only 1 sample
test_batch = DataProto.from_single_dict(batch_dict)

batch_keys = ["input_ids", "attention_mask", "position_ids"]
non_tensor_keys = ["raw_prompt_ids"]
test_gen_batch = test_batch.pop(batch_keys=batch_keys, non_tensor_batch_keys=non_tensor_keys)
test_gen_batch.meta_info.update({
    "do_sample": cfg.actor_rollout_ref.rollout.val_kwargs.do_sample,
    "temperature": cfg.actor_rollout_ref.rollout.val_kwargs.temperature,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
})

if "raw_prompt" not in test_gen_batch.non_tensor_batch:
    decoded = tokenizer.batch_decode(
        test_gen_batch.non_tensor_batch["raw_prompt_ids"],
        skip_special_tokens=False,
    )
    test_gen_batch.non_tensor_batch["raw_prompt"] = [
        [{"role": "user", "content": text}] for text in decoded
    ]

if "data_source" not in test_gen_batch.non_tensor_batch:
    test_gen_batch.non_tensor_batch["data_source"] = ["craftext"] * len(test_gen_batch)

rollout = trainer.traj_collector.multi_turn_loop(
    gen_batch=test_gen_batch,
    actor_rollout_wg=trainer.actor_rollout_wg,
    envs=trainer.val_envs,
    is_train=False,
)
trajectory = to_list_of_dict(rollout)[0]     # first/only env in the batch

In [ ]:
traj = to_list_of_dict(rollout)

In [ ]:
print(len(traj))
print(traj[0].keys()) 

In [ ]:
action_ids =traj[0]['responses']
print(action_ids)

text_actions = tokenizer.batch_decode(action_ids, skip_special_tokens=True)
print(text_actions)

print(traj[0]['is_action_valid'])
# print(traj[0]['prompts'])
# print(traj[0]['responses'])
# print(traj[0]['rewards'])
# print(traj[0]['active_masks'])
# print(traj[0]['anchor_obs'])


In [ ]:
summary = {}
for key, value in trajectory.items():
    if isinstance(value, (list, tuple)):
        summary[key] = (type(value[0]).__name__, len(value))
    else:
        summary[key] = (type(value).__name__, None)
summary


In [ ]:
import pandas as pd

step_count = len(trajectory["responses"])
print(step_count)

records = []
for step in range(step_count):
    obs_text = trajectory["anchor_obs"][step]
    action_ids = trajectory["responses"][step]
    action_ids = action_ids.tolist() if hasattr(action_ids, "tolist") else action_ids
    action_text = tokenizer.decode(action_ids, skip_special_tokens=True).strip()
    reward = float(trajectory["rewards"][step])
    done = not bool(trajectory["active_masks"][step])
    records.append({
        "step": step,
        "observation": obs_text,
        "action": action_text,
        "reward": reward,
        "done": done,
    })
traj_df = pd.DataFrame(records)
traj_df.head()


In [ ]:
step = 0
print("Observation:\n", traj_df.loc[step, "observation"])
print("\nAction:\n", traj_df.loc[step, "action"])
print("\nReward:", traj_df.loc[step, "reward"], " Done:", traj_df.loc[step, "done"])


In [ ]:
trainer.actor_rollout_wg.shutdown()
if trainer.use_critic:
    trainer.critic_wg.shutdown()
trainer.resource_pool_manager.destroy()
ray.shutdown()